In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 37.0 MB/s  0:00:01


In [ ]:
import os
import shutil
from google.colab import drive

drive.mount("/content/drive")

# Set up your persistent directory on Drive
CHECKPOINT_DIR = "/content/drive/MyDrive/sprog_checkpoints/tinystories_run_v3"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BIN_PATH = os.path.join(CHECKPOINT_DIR, "train.bin")
TOKEN_PROGRESS_PATH = os.path.join(CHECKPOINT_DIR, "tokenization_progress.txt")
CKPT_PATH = os.path.join(CHECKPOINT_DIR, "latest_ckpt.pt")

Mounted at /content/drive


In [ ]:
import os
import numpy as np
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer
from tqdm.auto import tqdm

# Load/initialize tokenizer
tokenizer = ByteLevelBPETokenizer()
# Ensure your tokenizer files are loaded or saved:
tokenizer = ByteLevelBPETokenizer.from_file(os.path.join(CHECKPOINT_DIR, "tinystories_run-vocab.json"), os.path.join(CHECKPOINT_DIR, "tinystories_run-merges.txt"))
eos_id = tokenizer.token_to_id("<|endoftext|>")
if eos_id is None:
    tokenizer.add_special_tokens(["<|endoftext|>"])
    eos_id = tokenizer.token_to_id("<|endoftext|>")

ds = load_dataset("roneneldan/TinyStories", split="train")
total_examples = len(ds)

# Check last processed index
start_idx = 0
if os.path.exists(TOKEN_PROGRESS_PATH) and os.path.exists(BIN_PATH):
    with open(TOKEN_PROGRESS_PATH, "r") as f:
        content = f.read().strip()
        if content.isdigit():
            start_idx = int(content)
            print(f"Resuming tokenization from example index {start_idx}/{total_examples}...")

CHUNK_SIZE = 10000  # Number of examples to process before flushing to Drive

if start_idx < total_examples:
    with open(BIN_PATH, "ab") as bin_file:  # Append binary mode
        buffer = []
        for i in range(start_idx, total_examples):
            text = ds[i]["text"]
            encoded = tokenizer.encode(text).ids
            encoded.append(eos_id)
            buffer.extend(encoded)

            # Flush chunk to Drive
            if (i + 1) % CHUNK_SIZE == 0 or (i + 1) == total_examples:
                chunk_arr = np.array(buffer, dtype=np.uint16)
                bin_file.write(chunk_arr.tobytes())
                bin_file.flush()
                os.fsync(bin_file.fileno())
                buffer = []

                # Update progress tracker atomically
                with open(TOKEN_PROGRESS_PATH + ".tmp", "w") as f:
                    f.write(str(i + 1))
                os.replace(TOKEN_PROGRESS_PATH + ".tmp", TOKEN_PROGRESS_PATH)
                print(f"Flushed up to index {i + 1}/{total_examples}")

    print("Tokenization and packing complete.")
else:
    print(f"Binary file is already complete ({total_examples}/{total_examples} examples).")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

data/train-00002-of-00004-a26307300439e9(…): reconstructing file:   0%|          |  0.00B /  246MB            

... (4 outputs left out)

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Resuming tokenization from example index 2119719/2119719...
Binary file is already complete (2119719/2119719 examples).


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * norm * self.weight


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv_proj = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv_proj(x).reshape(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # PyTorch Native Scaled Dot-Product Attention (uses FlashAttention/cutlass under the hood)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class MLP(nn.Module):
    def __init__(self, n_embd: int):
        super().__init__()
        hidden_dim = 4 * n_embd
        # SwiGLU-style feedforward
        self.gate_proj = nn.Linear(n_embd, hidden_dim, bias=False)
        self.up_proj = nn.Linear(n_embd, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, n_embd, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class TransformerBlock(nn.Module):
    def __init__(self, n_embd: int, n_head: int):
        super().__init__()
        self.attn_norm = RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.mlp_norm = RMSNorm(n_embd)
        self.mlp = MLP(n_embd)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        x = x + self.mlp(self.mlp_norm(x))
        return x


class CustomGPT(nn.Module):
    def __init__(self, vocab_size=32000, n_layer=8, n_head=8, n_embd=512, block_size=512):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([TransformerBlock(n_embd, n_head) for _ in range(n_layer)])
        self.norm = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying (reduces parameter footprint significantly)
        self.tok_emb.weight = self.lm_head.weight

    def forward(self, idx):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        logits = self.lm_head(x)
        return logits

In [ ]:
# import os
# import math
# import shutil
# import torch
# import torch.nn as nn
# import numpy as np
# import bitsandbytes as bnb
# from torch.optim.lr_scheduler import LambdaLR

# # ----------------- Configuration & Paths -----------------
# LOCAL_BIN_PATH = "/content/train.bin"

# total_steps = 50000
# warmup_steps = 2000
# min_lr_ratio = 0.1
# gradient_accumulation_steps = 4
# save_interval = 500

# # ----------------- Setup & Local Copy -----------------
# if not os.path.exists(LOCAL_BIN_PATH) and os.path.exists(BIN_PATH):
#     shutil.copy(BIN_PATH, LOCAL_BIN_PATH)

# data = np.memmap(LOCAL_BIN_PATH, dtype=np.uint16, mode="r")

# def get_batch(batch_size=8, block_size=512):
#     ix = torch.randint(len(data) - block_size, (batch_size,))
#     x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
#     y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
#     return x.cuda(), y.cuda()

# # ----------------- Serialization Helpers -----------------
# def get_clean_bnb_optimizer_state(opt):
#     """
#     Sanitizes bitsandbytes optimizer state.
#     Extracts purely numerical tensors and standard primitives, stripping
#     internal C++/CUDA page pointers and unmanaged memory references.
#     """
#     raw_state = opt.state_dict()
#     clean_state = {"state": {}, "param_groups": raw_state["param_groups"]}

#     for param_id, state_dict in raw_state["state"].items():
#         clean_state["state"][param_id] = {}
#         for key, value in state_dict.items():
#             if isinstance(value, torch.Tensor):
#                 # Detach, ensure contiguous memory, and move to CPU
#                 clean_state["state"][param_id][key] = value.detach().contiguous().cpu().clone()
#             elif isinstance(value, (int, float, bool)):
#                 clean_state["state"][param_id][key] = value

#     return clean_state

# # ----------------- Model & Optimizer Initialization -----------------
# # Replace CustomGPT with your model class definition
# base_model = CustomGPT(vocab_size=32000, n_layer=8, n_head=8, n_embd=512, block_size=512).cuda()
# optimizer = bnb.optim.PagedAdamW8bit(base_model.parameters(), lr=6e-4, weight_decay=0.1)
# scaler = torch.cuda.amp.GradScaler()

# # ----------------- Learning Rate Schedule -----------------
# def get_lr_lambda(current_step: int):
#     if current_step < warmup_steps:
#         return float(current_step) / float(max(1, warmup_steps))
#     progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
#     progress = min(max(progress, 0.0), 1.0)
#     cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
#     return min_lr_ratio + (1.0 - min_lr_ratio) * cosine_decay

# scheduler = LambdaLR(optimizer, lr_lambda=get_lr_lambda)

# # ----------------- Checkpoint Restoration -----------------
# start_step = 0
# if os.path.exists(CKPT_PATH):
#     print(f"Loading checkpoint from {CKPT_PATH}...")
#     checkpoint = torch.load(CKPT_PATH, map_location="cpu")

#     base_model.load_state_dict(checkpoint["model_state_dict"])
#     base_model.cuda()

#     # Load cleaned state into the optimizer
#     optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

#     # Re-register tensors as CUDA contiguous blocks so bitsandbytes reallocates pages properly
#     for state in optimizer.state.values():
#         for k, v in state.items():
#             if isinstance(v, torch.Tensor):
#                 state[k] = v.cuda().contiguous()

#     scaler.load_state_dict(checkpoint["scaler_state_dict"])

#     if "scheduler_state_dict" in checkpoint:
#         scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

#     start_step = checkpoint["step"] + 1
#     print(f"Resumed at step {start_step} with learning rate: {scheduler.get_last_lr()[0]:.6e}")
# else:
#     print("Starting fresh training run.")

# model = torch.compile(base_model)

# # ----------------- Training Loop -----------------
# model.train()
# for step in range(start_step, total_steps):
#     optimizer.zero_grad(set_to_none=True)
#     accum_loss = 0.0

#     for micro_step in range(gradient_accumulation_steps):
#         x, y = get_batch(batch_size=8, block_size=512)
#         with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
#             logits = model(x)
#             loss = nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
#             loss = loss / gradient_accumulation_steps

#         scaler.scale(loss).backward()
#         accum_loss += loss.item()

#     scaler.unscale_(optimizer)
#     torch.nn.utils.clip_grad_norm_(base_model.parameters(), 1.0)

#     scale_before = scaler.get_scale()
#     scaler.step(optimizer)
#     scaler.update()
#     scale_after = scaler.get_scale()

#     # Only step scheduler if gradients were not skipped due to Inf/NaN
#     if scale_before <= scale_after:
#         scheduler.step()

#     if (step + 1) % 100 == 0:
#         current_lr = scheduler.get_last_lr()[0]
#         print(f"Step {step + 1}/{total_steps} | Loss: {accum_loss:.4f} | LR: {current_lr:.6e}")

#     # ----------------- Safe Checkpointing -----------------
#     if (step + 1) % save_interval == 0 or (step + 1) == total_steps:
#         ckpt_data = {
#             "step": step,
#             "model_state_dict": base_model.state_dict(),
#             "optimizer_state_dict": get_clean_bnb_optimizer_state(optimizer),
#             "scaler_state_dict": scaler.state_dict(),
#             "scheduler_state_dict": scheduler.state_dict(),
#             "loss": accum_loss,
#         }
#         tmp_ckpt_path = CKPT_PATH + ".tmp"
#         torch.save(ckpt_data, tmp_ckpt_path)
#         os.replace(tmp_ckpt_path, CKPT_PATH)
#         print(f"Checkpoint successfully saved at step {step + 1}.")

In [ ]:
import os
import math
import random
import shutil
import numpy as np
import torch
import torch.nn as nn
import bitsandbytes as bnb

from torch.optim.lr_scheduler import LambdaLR

# ============================================================
# CONFIGURATION
# ============================================================

BIN_PATH = BIN_PATH
LOCAL_BIN_PATH = "/content/train.bin"

CKPT_DIR = CHECKPOINT_DIR
CKPT_PATH = os.path.join(CKPT_DIR, "latest.ckpt")

VOCAB_SIZE = 32000
N_LAYER = 8
N_HEAD = 8
N_EMBD = 512
BLOCK_SIZE = 512

BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4

TOTAL_STEPS = 50000
WARMUP_STEPS = 2000
SAVE_INTERVAL = 500
LOG_INTERVAL = 100

BASE_LR = 6e-4
WEIGHT_DECAY = 0.1
MIN_LR_RATIO = 0.1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# This script uses CUDA AMP and bitsandbytes PagedAdamW8bit.
if DEVICE != "cuda":
    raise RuntimeError(
        "This training script requires a CUDA GPU. "
        "Select a GPU runtime in Google Colab."
    )

os.makedirs(CKPT_DIR, exist_ok=True)

torch.manual_seed(1337)
np.random.seed(1337)
random.seed(1337)
torch.cuda.manual_seed_all(1337)

print("Device:", DEVICE)
print("Checkpoint:", CKPT_PATH)

# ============================================================
# DATASET
# ============================================================

if not os.path.exists(LOCAL_BIN_PATH):
    if not os.path.exists(BIN_PATH):
        raise FileNotFoundError(
            f"Token file not found:\n{BIN_PATH}\n"
            "Check BIN_PATH and mount Google Drive."
        )

    print("Copying token file to local storage...")
    shutil.copy2(BIN_PATH, LOCAL_BIN_PATH)

data = np.memmap(
    LOCAL_BIN_PATH,
    dtype=np.uint16,
    mode="r",
)

if len(data) <= BLOCK_SIZE:
    raise ValueError(
        f"Dataset contains {len(data)} tokens, "
        f"but BLOCK_SIZE is {BLOCK_SIZE}."
    )

print(f"Loaded token dataset: {len(data):,} tokens")


def get_batch(batch_size=BATCH_SIZE, block_size=BLOCK_SIZE):
    # Sample starting positions on the CPU.
    ix = torch.randint(
        low=0,
        high=len(data) - block_size,
        size=(batch_size,),
    ).tolist()

    x = torch.stack([
        torch.from_numpy(
            data[i:i + block_size].astype(np.int64)
        )
        for i in ix
    ])

    y = torch.stack([
        torch.from_numpy(
            data[i + 1:i + 1 + block_size].astype(np.int64)
        )
        for i in ix
    ])

    return x.to(DEVICE), y.to(DEVICE)


# ============================================================
# LEARNING RATE SCHEDULE
# ============================================================

def get_lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return current_step / max(1, WARMUP_STEPS)

    progress = (
        (current_step - WARMUP_STEPS)
        / max(1, TOTAL_STEPS - WARMUP_STEPS)
    )

    progress = min(max(progress, 0.0), 1.0)

    cosine_decay = 0.5 * (
        1.0 + math.cos(math.pi * progress)
    )

    return MIN_LR_RATIO + (
        1.0 - MIN_LR_RATIO
    ) * cosine_decay


# ============================================================
# CHECKPOINT HELPERS
# ============================================================

def cpu_rng_state():
    return torch.get_rng_state().cpu().clone()


def cuda_rng_states():
    if torch.cuda.is_available():
        return [
            state.cpu().clone()
            for state in torch.cuda.get_rng_state_all()
        ]
    return None


def save_checkpoint(
    path,
    step,
    model,
    optimizer,
    scheduler,
    scaler,
    loss,
):
    """
    Save a complete training checkpoint atomically.

    Call this after the optimizer/scaler/scheduler update,
    at the end of an optimizer step.
    """

    checkpoint = {
        "format_version": 1,
        "step": int(step),

        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),

        "loss": float(loss),

        "config": {
            "vocab_size": VOCAB_SIZE,
            "n_layer": N_LAYER,
            "n_head": N_HEAD,
            "n_embd": N_EMBD,
            "block_size": BLOCK_SIZE,
            "batch_size": BATCH_SIZE,
            "gradient_accumulation_steps":
                GRADIENT_ACCUMULATION_STEPS,
            "total_steps": TOTAL_STEPS,
            "warmup_steps": WARMUP_STEPS,
            "base_lr": BASE_LR,
            "weight_decay": WEIGHT_DECAY,
            "min_lr_ratio": MIN_LR_RATIO,
            "data_path": BIN_PATH,
        },

        # Random-number-generator states for resuming.
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": cpu_rng_state(),
        "cuda_rng_state_all": cuda_rng_states(),
    }

    tmp_path = path + ".tmp"

    try:
        torch.save(checkpoint, tmp_path)

        # Replace the previous checkpoint only after saving
        # the new checkpoint successfully.
        os.replace(tmp_path, path)

    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


def restore_rng_states(checkpoint):
    if "python_rng_state" in checkpoint:
        random.setstate(checkpoint["python_rng_state"])

    if "numpy_rng_state" in checkpoint:
        np.random.set_state(checkpoint["numpy_rng_state"])

    if checkpoint.get("torch_rng_state") is not None:
        state = checkpoint["torch_rng_state"]
        state = state.detach().cpu().to(torch.uint8)
        torch.set_rng_state(state)

    saved_cuda_states = checkpoint.get("cuda_rng_state_all")

    if saved_cuda_states is not None and torch.cuda.is_available():
        current_device_count = torch.cuda.device_count()

        if len(saved_cuda_states) == current_device_count:
            states = [
                state.detach().cpu().to(torch.uint8)
                for state in saved_cuda_states
            ]
            torch.cuda.set_rng_state_all(states)
        else:
            print(
                "Warning: CUDA device count differs from the "
                "checkpoint. CUDA RNG state was not restored."
            )


def load_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
):
    """
    Restore model and training state.

    Returns the next step to run, or 0 if no checkpoint exists.
    """

    if not os.path.isfile(path):
        print("No checkpoint found. Starting from scratch.")
        return 0

    print(f"Loading checkpoint: {path}")

    # Only load checkpoints you created and trust.
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    # Restore model parameters.
    model.load_state_dict(checkpoint["model_state_dict"])

    # Restore optimizer state.
    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    # Do not manually move every bitsandbytes state tensor.
    # Let bitsandbytes manage its optimizer state.
    scaler.load_state_dict(
        checkpoint.get("scaler_state_dict", {})
    )

    if "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(
            checkpoint["scheduler_state_dict"]
        )

    restore_rng_states(checkpoint)

    saved_step = int(checkpoint["step"])
    next_step = saved_step + 1

    print(f"Checkpoint step: {saved_step + 1}")
    print(f"Resuming at loop step: {next_step}")
    print(
        "Restored learning rate:",
        f"{scheduler.get_last_lr()[0]:.6e}",
    )

    del checkpoint

    return next_step


# ============================================================
# MODEL, OPTIMIZER, SCALER, SCHEDULER
# ============================================================

# CustomGPT must be defined earlier in the notebook.
base_model = CustomGPT(
    vocab_size=VOCAB_SIZE,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_embd=N_EMBD,
    block_size=BLOCK_SIZE,
).to(DEVICE)

optimizer = bnb.optim.PagedAdamW8bit(
    base_model.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True,
)

scheduler = LambdaLR(
    optimizer,
    lr_lambda=get_lr_lambda,
)

# Restore the uncompiled model first.
start_step = load_checkpoint(
    CKPT_PATH,
    base_model,
    optimizer,
    scheduler,
    scaler,
)

# Compile only after checkpoint restoration.
try:
    model = torch.compile(base_model)
    print("torch.compile enabled.")
except Exception as exc:
    print(f"torch.compile unavailable; using eager mode: {exc}")
    model = base_model


# ============================================================
# TRAINING
# ============================================================

model.train()

for step in range(start_step, TOTAL_STEPS):
    optimizer.zero_grad(set_to_none=True)
    accumulated_loss = 0.0

    for micro_step in range(GRADIENT_ACCUMULATION_STEPS):
        x, y = get_batch()

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            logits = model(x)

            loss = nn.functional.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1),
            )

            loss = loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(loss).backward()
        accumulated_loss += loss.detach().item()

    scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(
        base_model.parameters(),
        max_norm=1.0,
    )

    scale_before = scaler.get_scale()

    scaler.step(optimizer)
    scaler.update()

    scale_after = scaler.get_scale()

    # If AMP skipped the optimizer update because of Inf/NaN,
    # don't advance the scheduler.
    if scale_after >= scale_before:
        scheduler.step()

    completed_step = step + 1

    if completed_step % LOG_INTERVAL == 0:
        current_lr = scheduler.get_last_lr()[0]

        print(
            f"Step {completed_step}/{TOTAL_STEPS} | "
            f"Loss: {accumulated_loss:.4f} | "
            f"LR: {current_lr:.6e} | "
            f"Scale: {scaler.get_scale():.1f}"
        )

    # Save only at optimizer-step boundaries.
    if (
        completed_step % SAVE_INTERVAL == 0
        or completed_step == TOTAL_STEPS
    ):
        save_checkpoint(
            path=CKPT_PATH,
            step=step,
            model=base_model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            loss=accumulated_loss,
        )

        print(
            f"Checkpoint saved at step {completed_step}: "
            f"{CKPT_PATH}"
        )

print("Training finished.")

Device: cuda
Checkpoint: /content/drive/MyDrive/sprog_checkpoints/tinystories_run_v3/latest.ckpt
Copying token file to local storage...
Loaded token dataset: 462,302,769 tokens
Loading checkpoint: /content/drive/MyDrive/sprog_checkpoints/tinystories_run_v3/latest.ckpt
Checkpoint step: 34500
Resuming at loop step: 34500
Restored learning rate: 1.875568e-04
torch.compile enabled.


W1028 14:46:26.171000 442 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


Step 34600/50000 | Loss: 1.5383 | LR: 1.860586e-04 | Scale: 1048576.0
Step 34700/50000 | Loss: 1.5972 | LR: 1.845666e-04 | Scale: 1048576.0
Step 34800/50000 | Loss: 1.4659 | LR: 1.830809e-04 | Scale: 1048576.0
Step 34900/50000 | Loss: 1.5767 | LR: 1.816014e-04 | Scale: 1048576.0
Step 35000/50000 | Loss: 1.5033 | LR: 1.801283e-04 | Scale: 1048576.0
Checkpoint saved at step 35000: /content/drive/MyDrive/sprog_checkpoints/tinystories_run_v3/latest.ckpt
Step 35100/50000 | Loss: 1.4520 | LR: 1.786616e-04 | Scale: 1048576.0
Step 35200/50000 | Loss: 1.7144 | LR: 1.772160e-04 | Scale: 524288.0
Step 35300/50000 | Loss: 1.4748 | LR: 1.757623e-04 | Scale: 524288.0
Step 35400/50000 | Loss: 1.4890 | LR: 1.743151e-04 | Scale: 524288.0
Step 35500/50000 | Loss: 1.5687 | LR: 1.728747e-04 | Scale: 524288.0
Checkpoint saved at step 35500: /content/drive/MyDrive/sprog_checkpoints/tinystories_run_v3/latest.ckpt
Step 35600/50000 | Loss: 1.4691 | LR: 1.714410e-04 | Scale: 524288.0
Step 35700/50000 | Loss: 1.